<a href="https://colab.research.google.com/github/draygada/DataSci-112-Final-Project---Diego-Raygada-Townsend-Miller/blob/main/DataSci_112_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import pandas as pd
import yfinance as yf
import numpy as np
import re
import torch
from transformers import pipeline as hf_pipeline
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from scipy.stats import norm
from concurrent.futures import ThreadPoolExecutor, as_completed
import datetime
import warnings
warnings.filterwarnings("ignore")

# **Reading in our Data**

We first scrape the S&P 500 to get the full list of tickers, then use that list to bulk download 10 years of historical price data for each company, along with SPY as a market benchmark, so everything’s ready for analysis.

In [ ]:
#Step 1: Get the S&P 500 tickers
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
response = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"}
)
tables = pd.read_html(response.text)
tickers = tables[0]["Symbol"].tolist()
tickers = [t.replace(".", "-") for t in tickers] # Contains a list of all tickers in the S&P 500

print(f"Loaded {len(tickers)} tickers")


# Step 2: Download 10 years (2016 - 2026) of stock price data for each ticket in the S&P 500
print("Downloading price history (bulk, 10y)...")

raw_all = yf.download(
    tickers,
    period="10y",
    progress=True,
    group_by="ticker"
)


# Step 3: Download the SPY Data to use as a benchmark
spy_raw = yf.download("SPY", period="10y", progress=False)
if isinstance(spy_raw.columns, pd.MultiIndex):
    spy_raw.columns = spy_raw.columns.get_level_values(0)
spy_close   = spy_raw["Close"].squeeze()
spy_log_ret = np.log(spy_close / spy_close.shift(1))


#Step 4: Helper function to clean individual ticker data
def get_ticker_data(raw_all, ticker):
    try:
        # check ticker exists
        if ticker in raw_all.columns.get_level_values(0):
            df = raw_all[ticker].copy()
        else:
            return None

        # flatten columns if multi-indexed
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        # remove duplicate columns
        df = df.loc[:, ~df.columns.duplicated()]

        # drop rows that are completely empty
        df = df.dropna(how="all")

        # require at least 1 year of data
        return df if len(df) >= 252 else None

    except Exception:
        return None

In [ ]:
display(spy_raw)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA

# make sure spy_close is a pandas Series
spy = spy_close.copy()

ma50 = spy.rolling(window=50).mean()
ma200 = spy.rolling(window=200).mean()

plt.figure()
plt.plot(spy.index, spy.values, label="SPY")
plt.plot(ma50.index, ma50.values, label="50-Day Moving Average")
plt.plot(ma200.index, ma200.values, label="200-Day Moving Average")

plt.title("SPY Price with 50-Day and 200-Day Moving Averages")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.grid(True)
plt.show()

# **Loading in Sentiment Model**
We load FinBERT, a finance-specific sentiment model, and use it to score recent stock headlines as positive, negative, or neutral. We also flag major catalyst events like earnings, FDA decisions, mergers, or macro releases, then use that sentiment to check whether the news direction lines up with a call or put trade.

In [ ]:
print("Loading FinBERT model...")
device     = 0 if torch.cuda.is_available() else -1
#Create a sentiment analysis/text-calssification pipeline using a version of BERT trained on financial text
finbert    = hf_pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    tokenizer="ProsusAI/finbert",
    device=device,
    truncation=True,
    max_length=512,
)
print(f"  FinBERT loaded on {'GPU' if device == 0 else 'CPU'}")

# Set of words/phrases that represent important stock-moving events
CATALYST_KEYWORDS = {
    "earnings", "quarterly results", "q1", "q2", "q3", "q4",
    "eps", "revenue report", "results",
    "fda", "pdufa", "approval", "clinical trial",
    "phase 3", "phase 2", "regulatory", "clearance",
    "merger", "acquisition", "takeover", "spinoff",
    "ipo", "secondary offering", "buyout", "going private",
    "fed decision", "fomc", "rate decision", "inflation data",
    "cpi", "jobs report", "gdp",
    "doj", "sec investigation", "class action", "settlement",
}

#Defining the main new sentiment function. Inputs a ticket
def analyze_news_sentiment(tkr_obj):
    """
    Fetches recent headlines via yfinance and scores them with FinBERT.
    Returns:
      sentiment_score : float in [-1, 1]
      has_catalyst    : bool
      catalyst_type   : str
      headline_count  : int
    """
    try:
        news = tkr_obj.news
        if not news:
            return 0.0, False, "", 0

        titles        = []
        catalyst_hits = []

        #Gets the first 10 articles about that ticker
        for article in news[:10]:
            title = (article.get("title", "") or
                     article.get("headline", "") or
                     article.get("content", {}).get("title", "") or "")
            if not title:
                continue
            titles.append(title)
            tl = title.lower()
            for kw in CATALYST_KEYWORDS:
                if kw in tl:
                    catalyst_hits.append(kw)

        if not titles:
            return 0.0, False, "", 0

        #Clasifies each headline as either positive, negative, or neutral with a provided confidence score
        results = finbert(titles, batch_size=8)

        scores = []
        for res in results:
            label = res["label"].lower()
            conf  = res["score"]
            if label == "positive":
                scores.append(conf)
            elif label == "negative":
                scores.append(-conf)
            else:
                scores.append(0.0)

        #Finds average sentiment score
        avg_sentiment = float(np.mean(scores)) if scores else 0.0
        has_catalyst  = len(catalyst_hits) > 0
        catalyst_type = catalyst_hits[0] if catalyst_hits else ""

        return avg_sentiment, has_catalyst, catalyst_type, len(scores)

    except Exception:
        return 0.0, False, "", 0


def sentiment_direction_aligned(sentiment_score, option_type):
    if abs(sentiment_score) < 0.1:
        return True
    return sentiment_score > 0 if option_type == "call" else sentiment_score < 0

# **Explanatory Data Analysis**
**1. Distribution of Daily Returns Across S&P 500 Stocks**

Most daily stock returns are small and clustered near zero, but extreme moves still occur, motivating the use of volatility-aware features and options-based analysis.

In [ ]:
# Distribution of daily returns across S&P 500 stocks (clipped for readability)
close_df = raw_all.xs("Close", axis=1, level=1)
all_daily_returns = close_df.pct_change().stack().dropna()

plt.figure(figsize=(10, 6))
plt.hist(all_daily_returns.clip(-0.1, 0.1), bins=100)

plt.title("Distribution of Daily Returns Across S&P 500 Stocks")
plt.xlabel("Daily Return")
plt.ylabel("Frequency")
plt.grid(True)
plt.show()

 **2. Implied VS Realized Volatility**

Rolling volatility is how much the market has been moving recently, measured over a sliding window of time. We see that the implicit volatility is always great than realized volatility as the market always retains a level of fear.

In [ ]:
vix = yf.download("^VIX", period="10y", progress=False)

if isinstance(vix.columns, pd.MultiIndex):
    vix.columns = vix.columns.get_level_values(0)

vix_close = vix["Close"].squeeze() / 100  # convert percent to decimal

rolling_vol_30 = spy_log_ret.rolling(30).std() * np.sqrt(252)

plt.figure(figsize=(10, 6))
plt.plot(
    rolling_vol_30.index,
    rolling_vol_30.values,
    label="Realized Vol (30d)"
)

plt.plot(
    vix_close.index,
    vix_close.values,
    label="Implied Vol (VIX)"
)

plt.title("Implied vs Realized Volatility (SPY)")
plt.xlabel("Date")
plt.ylabel("Volatility")
plt.legend()
plt.grid(True)
plt.show()

# **Horizon Buckets**
We define a set of standard time horizons, from 1 week to 1 year, and map each one to a number of days plus relevant volatility windows. Then, given a contract’s days to expiration, we choose the horizon bucket that is closest to it so later calculations use the most appropriate time frame.

In [ ]:
#We group option expiration dates into horizons
HORIZONS = {
    "1w": {
        "days": 7,
        "vol_windows": [5, 10],
    },
    "2w": {
        "days": 14,
        "vol_windows": [10, 20],
    },
    "1m": {
        "days": 30,
        "vol_windows": [20, 60],
    },
    "2m": {
        "days": 60,
        "vol_windows": [30, 60],
    },
    "3m": {
        "days": 90,
        "vol_windows": [60, 120],
    },
    "6m": {
        "days": 180,
        "vol_windows": [60, 120],
    },
    "1y": {
        "days": 365,
        "vol_windows": [120, 252],
    },
}


def horizon_for_dte(dte):
    return min(HORIZONS, key=lambda h: abs(HORIZONS[h]["days"] - dte))

# **Feature Engineering**
This function builds the input features for the model using each stock’s price and volume history. It creates momentum, volatility, mean-reversion, relative-strength, beta, price-position, and trend-based indicators, then defines the prediction target as the stock’s forward return over the selected horizon.

In [ ]:
def build_features(data, horizon_days, vol_windows, spy_log_ret):
    data = data.copy()
    close = data["Close"].squeeze()

    data["log_ret"] = np.log(close / close.shift(1))

    #Gets returns over the last 5, 20, 60 days
    data["ret_5"] = close.pct_change(5)
    data["ret_20"] = close.pct_change(20)
    data["ret_60"] = close.pct_change(60)

    feature_cols = ["ret_5", "ret_20", "ret_60"]

    #For each vol window, it measures a stocks volatility
    for w in vol_windows:
        col = f"vol_{w}"
        data[col] = data["log_ret"].rolling(w).std()
        feature_cols.append(col)

    #Measures how far today’s price is from its rolling average, in standard deviation units
    roll60 = close.rolling(60)
    data["z_score_60"] = (close - roll60.mean()) / roll60.std()

    roll252 = close.rolling(252)
    data["z_score_252"] = (close - roll252.mean()) / roll252.std()

    #Measure distance from 52-week high and low
    data["pct_from_52w_high"] = close / close.rolling(252).max() - 1
    data["pct_from_52w_low"] = close / close.rolling(252).min() - 1

    #Compares short-term volatility to long-term volatility to see recent change in volatility
    short_vol_col = f"vol_{vol_windows[0]}"
    long_vol_col = f"vol_{vol_windows[-1]}"
    data["vol_ratio"] = data[short_vol_col] / data[long_vol_col]

    feature_cols += [
        "z_score_60",
        "z_score_252",
        "pct_from_52w_high",
        "pct_from_52w_low",
        "vol_ratio",
    ]

    spy_aligned = spy_log_ret.reindex(close.index).ffill()

    #Computes the stock's returns to SPY over 5,20,60 days indicating whether the stock is outperforming or underperforming the market.
    data["rs_5"] = data["log_ret"].rolling(5).sum() - spy_aligned.rolling(5).sum()
    data["rs_20"] = data["log_ret"].rolling(20).sum() - spy_aligned.rolling(20).sum()
    data["rs_60"] = data["log_ret"].rolling(60).sum() - spy_aligned.rolling(60).sum()

    #Calculates the stock’s 60-day beta relative to SPY
    cov = data["log_ret"].rolling(60).cov(spy_aligned)
    var = spy_aligned.rolling(60).var()
    data["beta_60"] = cov / var.replace(0, np.nan)

    #Check if stock is above its 200-day moving average
    data["above_200ma"] = (close > close.rolling(200).mean()).astype(float)

    feature_cols += ["rs_5", "rs_20", "rs_60", "beta_60", "above_200ma"]

    #Use high and low prices to measure daily position
    high = data["High"].squeeze()
    low = data["Low"].squeeze()
    day_range = (high - low).replace(0, np.nan)

    data["close_position"] = (close - low) / day_range
    data["close_pos_5"] = data["close_position"].rolling(5).mean()
    data["close_pos_20"] = data["close_position"].rolling(20).mean()
    data["hl_range_pct"] = (high - low) / close

    feature_cols += ["close_pos_5", "close_pos_20", "hl_range_pct"]

    #Compute short-term autocorrelation
    data["autocorr_10"] = data["log_ret"].rolling(20).apply(
        lambda x: pd.Series(x).autocorr(lag=1) if len(x) > 1 else 0,
        raw=False,
    )

    #Compute exponentially weighted momentum signals
    data["ewm_ret_fast"] = close.ewm(span=10).mean().pct_change()
    data["ewm_ret_slow"] = close.ewm(span=30).mean().pct_change()
    data["ewm_diff"] = data["ewm_ret_fast"] - data["ewm_ret_slow"]

    feature_cols += ["autocorr_10", "ewm_ret_fast", "ewm_ret_slow", "ewm_diff"]

    #Add a volume-based feature if volume exists
    if "Volume" in data.columns:
        vol = data["Volume"].squeeze()
        daily_ret = close.pct_change()
        vol_signed = vol * np.sign(daily_ret)
        data["obv_ratio"] = vol_signed.rolling(20).mean() / vol.rolling(20).mean()
        feature_cols.append("obv_ratio")

    data["target"] = close.shift(-horizon_days) / close - 1
    data = data.dropna(subset=feature_cols + ["target"])

    return data, feature_cols

# **Model Training**
This function trains a gradient boosting model on the engineered features to predict the stock’s forward return. It then uses the latest row of data to generate a prediction, clips that prediction to a realistic historical range, and also returns the stock’s most recent realized volatility.

In [ ]:
#Trains the prediction model for one stock and one time horizon
def train_model(data, feature_cols):
    X = data[feature_cols]
    y = data["target"]
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("gb", GradientBoostingRegressor(
            n_estimators=50,
            learning_rate=0.01,
            max_depth=3,
            subsample=0.8,
            random_state=42,
        )),
    ])
    model.fit(X, y)
    #Takes the last row of features and asks the trained model for a predicted future return
    predicted_return = model.predict(X.iloc[-1:])[0]
    hist_95th        = float(np.percentile(np.abs(y), 95))
    #Caps the return at the 95th percentile
    predicted_return = float(np.clip(predicted_return, -hist_95th, hist_95th))
    short_vol_col    = [c for c in feature_cols if c.startswith("vol_")][0]
    realized_vol     = float(data[short_vol_col].iloc[-1]) * np.sqrt(252)
    return predicted_return, realized_vol

# **Black-Scholes Helper Functions**
These helper functions use Black-Scholes math to estimate option prices, back out implied volatility from market premiums, calculate the probability that an option finishes in the money, and convert implied volatility into the market’s expected price move over the option’s time

In [ ]:
# Black-Scholes Helper Functions
def bs_price_vec(S, K, T, r, sigma, option_type):
    with np.errstate(divide="ignore", invalid="ignore"):
        d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)

        if option_type == "call":
            price = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
        else:
            price = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

    return np.where(np.isfinite(price), price, 0.0)


def implied_vol_vec(S, K, T, r, market_prices, option_type):
    market_prices = np.asarray(market_prices, dtype=float)
    sigma = np.full_like(market_prices, 0.3)

    intrinsic = (
        np.maximum(S - K, 0)
        if option_type == "call"
        else np.maximum(K - S, 0)
    )

    valid = (market_prices > 0) & (market_prices > intrinsic) & (T > 0)

    for _ in range(200):
        if not valid.any():
            break

        price = bs_price_vec(S, K, T, r, sigma, option_type)

        d1 = np.where(
            valid & (sigma > 0),
            (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T)),
            0.0,
        )

        vega = S * norm.pdf(d1) * np.sqrt(T)
        diff = price - market_prices

        update = valid & (np.abs(vega) > 1e-10)
        step = np.clip(
            np.where(update, diff / vega, 0.0),
            -0.5 * sigma,
            0.5 * sigma,
        )

        sigma = np.clip(sigma - step, 1e-6, 10.0)
        valid = valid & (np.abs(diff) > 1e-6)

    final_price = bs_price_vec(S, K, T, r, sigma, option_type)
    bad = ~np.isfinite(sigma) | (np.abs(final_price - market_prices) > 0.01)

    return np.where(bad, np.nan, sigma)


def itm_probability(S, K, T, r, sigma, option_type):
    if sigma <= 0 or T <= 0:
        return 0.0

    d2 = (np.log(S / K) + (r - 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    return norm.cdf(d2) if option_type == "call" else norm.cdf(-d2)


def implied_move(S, iv, T):
    return S * iv * np.sqrt(T)

# **Earnings Helper Function**
This helper function checks whether a stock has an earnings announcement scheduled before a given option expiration date, and it also estimates the typical size of past earnings-related moves using recent earnings surprise data.


In [ ]:
def get_earnings_info(tkr, today, exp_date):
    try:
        cal = tkr.calendar
        if isinstance(cal, pd.DataFrame):
            earnings_dates = pd.to_datetime(cal.loc["Earnings Date"].dropna().values)
        elif isinstance(cal, dict) and "Earnings Date" in cal:
            earnings_dates = pd.to_datetime([cal["Earnings Date"]])
        else:
            return False, 0.0
        upcoming     = [d.date() for d in earnings_dates if today < d.date() <= exp_date]
        has_earnings = len(upcoming) > 0
        hist         = tkr.earnings_dates
        avg_move     = 0.0
        if hist is not None and "Surprise(%)" in hist.columns:
            surprises = hist["Surprise(%)"].dropna().head(8)
            if len(surprises) > 0:
                avg_move = float(surprises.abs().mean()) / 100
        return has_earnings, avg_move
    except Exception:
        return False, 0.0


# **Composite Signal Score Function**
This function combines several signals into one overall score by rewarding options with a stronger predicted move relative to the market-implied move, sentiment that supports the trade direction, the presence of a major catalyst, and a higher expected return on premium.

In [ ]:
def compute_signal_score(
    move_ratio,
    sentiment_score,
    has_catalyst,
    sentiment_aligned,
    expected_rop,
):
    mr_norm = min(move_ratio / 5.0, 1.0)
    sent_bonus = abs(sentiment_score) * 0.2 if sentiment_aligned else 0.0
    cat_bonus = 0.15 if has_catalyst else 0.0
    rop_norm = min(max(expected_rop, 0) / 2.0, 1.0) * 0.10

    return mr_norm + sent_bonus + cat_bonus + rop_norm

# **Individual Ticker Processing**
This function runs the full screening process for one stock. It pulls the stock’s price history and option chain, matches each expiration to the closest time horizon, generates a model-based return forecast, incorporates sentiment and earnings context, computes option metrics like implied volatility and expected profit, filters out weak contracts, and returns the strongest call or put opportunities for that ticker.

In [ ]:
#Per-Ticker Proccessing
MIN_EXTRINSIC_PCT = 0.005
MIN_EXPECTED_RETURN = 0.10
today = datetime.date.today()
risk_free = 0.04


def process_ticker(ticker):
    rows = []
    try:
        data_raw = get_ticker_data(raw_all, ticker)
        if data_raw is None:
            return rows

        S0 = float(data_raw["Close"].dropna().iloc[-1])
        tkr = yf.Ticker(ticker)

        expirations = tkr.options
        if not expirations:
            return rows

        expirations_to_screen = [
            e for e in expirations
            if 1 <= (datetime.date.fromisoformat(e) - today).days <= 365
        ]
        if not expirations_to_screen:
            return rows

        sentiment_score, has_catalyst, catalyst_type, headline_count = \
            analyze_news_sentiment(tkr)

        furthest_exp = datetime.date.fromisoformat(expirations_to_screen[-1])
        has_earnings_any, avg_earnings_move = get_earnings_info(tkr, today, furthest_exp)

        model_cache = {}
        earnings_cache = {}

        for exp in expirations_to_screen:
            exp_date = datetime.date.fromisoformat(exp)
            days_to_exp = max((exp_date - today).days, 1)
            T = days_to_exp / 365

            bucket_key = horizon_for_dte(days_to_exp)
            bucket = HORIZONS[bucket_key]

            if bucket_key not in model_cache:
                data, feature_cols = build_features(
                    data_raw,
                    bucket["days"],
                    bucket["vol_windows"],
                    spy_log_ret,
                )

                if len(data) < 100:
                    model_cache[bucket_key] = None
                    continue

                predicted_return, realized_vol = train_model(data, feature_cols)
                model_cache[bucket_key] = (predicted_return, realized_vol)

            cached = model_cache.get(bucket_key)
            if cached is None:
                continue

            predicted_return, realized_vol = cached
            expected_ST = S0 * (1 + predicted_return)
            pred_move_up = S0 * predicted_return
            pred_move_down = S0 * abs(predicted_return)

            if exp not in earnings_cache:
                earnings_cache[exp] = get_earnings_info(tkr, today, exp_date)

            has_earnings, avg_move = earnings_cache[exp]

            chain = tkr.option_chain(exp)

            for option_type, df in zip(["call", "put"], [chain.calls, chain.puts]):
                df = df[np.abs(df["strike"] - S0) / S0 < 0.1].copy()
                if df.empty:
                    continue

                if option_type == "call" and predicted_return <= 0:
                    continue
                if option_type == "put" and predicted_return >= 0:
                    continue

                sent_aligned = sentiment_direction_aligned(sentiment_score, option_type)

                strikes = df["strike"].values
                mid = ((df["bid"] + df["ask"]) / 2).values
                last = (
                    df["lastPrice"].values
                    if "lastPrice" in df.columns
                    else np.zeros_like(mid)
                )
                premiums = np.where(mid > 0, mid, last)
                ivs = implied_vol_vec(S0, strikes, T, risk_free, premiums, option_type)

                for i, (_, row) in enumerate(df.iterrows()):
                    K = row["strike"]
                    premium = premiums[i]
                    iv = ivs[i]

                    if premium <= 0 or not np.isfinite(iv):
                        continue

                    mkt_implied_move = implied_move(S0, iv, T)
                    pred_move = pred_move_up if option_type == "call" else pred_move_down
                    move_ratio = pred_move / mkt_implied_move if mkt_implied_move > 0 else 0

                    prob_itm = itm_probability(S0, K, T, risk_free, iv, option_type)
                    payoff = (
                        max(expected_ST - K, 0)
                        if option_type == "call"
                        else max(K - expected_ST, 0)
                    )
                    expected_profit = prob_itm * payoff - premium
                    expected_rop = expected_profit / premium

                    intrinsic = (
                        max(S0 - K, 0)
                        if option_type == "call"
                        else max(K - S0, 0)
                    )

                    if (premium - intrinsic) < S0 * MIN_EXTRINSIC_PCT:
                        continue

                    if expected_rop < MIN_EXPECTED_RETURN:
                        continue

                    signal_score = compute_signal_score(
                        move_ratio,
                        sentiment_score,
                        has_catalyst,
                        sent_aligned,
                        expected_rop,
                    )

                    rows.append({
                        "Ticker": ticker,
                        "Type": option_type,
                        "Expiration": exp,
                        "Days_To_Exp": days_to_exp,
                        "Horizon_Bucket": bucket_key,
                        "Current_Price": round(S0, 2),
                        "Predicted_Price": round(expected_ST, 2),
                        "Predicted_Return": round(predicted_return, 4),
                        "Strike": K,
                        "Premium": round(premium, 4),
                        "Implied_Vol": round(iv, 4),
                        "Realized_Vol": round(realized_vol, 4),
                        "Implied_Move": round(mkt_implied_move, 2),
                        "Predicted_Move": round(pred_move, 2),
                        "Move_Ratio": round(move_ratio, 4),
                        "Sentiment": round(sentiment_score, 3),
                        "Sent_Aligned": sent_aligned,
                        "Has_Catalyst": has_catalyst,
                        "Catalyst_Type": catalyst_type,
                        "Headlines": headline_count,
                        "Has_Earnings": has_earnings,
                        "Avg_Earnings_Move": round(avg_move, 4),
                        "Signal_Score": round(signal_score, 4),
                        "Prob_ITM": round(prob_itm, 4),
                        "Expected_Profit": round(expected_profit, 4),
                        "Expected_ROP": round(expected_rop, 4),
                    })

    except Exception as e:
        print(f"Skipping {ticker}: {type(e).__name__}: {e}")

    return rows

# **Final Execution**
This block runs the option screener across all S&P 500 tickers in parallel, collects the strongest contracts, ranks them by signal score, and saves the top opportunities.

In [ ]:
print("Screening options chains...")

results = []

with ThreadPoolExecutor(max_workers=8) as executor:
    futures = {executor.submit(process_ticker, t): t for t in tickers}

    for i, future in enumerate(as_completed(futures)):
        results.extend(future.result())

        if (i + 1) % 50 == 0:
            print(f"  {i + 1}/{len(tickers)} tickers done, {len(results)} options so far")

if not results:
    print("No results found.")

else:
    results_df = pd.DataFrame(results)
    top_opportunities = results_df.sort_values("Signal_Score", ascending=False).head(25)

    print("\nTop 25 Options (ranked by Signal Score):\n")
    print(top_opportunities.to_string(index=False))

    top_opportunities.to_csv("top_options.csv", index=False)
    print("\nSaved to top_options.csv")

In [ ]:
total_top_opportunities = results_df.sort_values("Implied_Move", ascending=False)
total_top_opportunities.to_csv("total_top_options.csv", index=False)

df = pd.read_csv("total_top_options.csv")
df.head(25)

In [ ]:
best_per_ticker = (
    df.sort_values("Expected_ROP", ascending=False)
    .groupby("Ticker")
    .first()
    .reset_index()
    .sort_values("Expected_ROP", ascending=False)
)

best_per_ticker

# **Data Analysis**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 7))

# Using regplot with lowess=True for the non-linear trend line
sns.regplot(
    x='Move_Ratio',
    y='Prob_ITM',
    data=best_per_ticker,
    scatter=False,
    lowess=True,
    color='red',
    label='Non-linear Trend (Lowess)'
)

# Overlaying the scatter points
sns.scatterplot(
    x='Move_Ratio',
    y='Prob_ITM',
    hue='Horizon_Bucket',
    size='Expected_ROP',
    sizes=(20, 200),
    alpha=0.6,
    data=best_per_ticker
)

plt.title('Non-linear Relationship: Prob_ITM vs. Move Ratio')
plt.xlabel('Move Ratio (Predicted Move / Implied Move)')
plt.ylabel('Probability ITM')
plt.axvline(1.0, color='gray', linestyle='--', label='Fairly Priced')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Filter for rows with catalysts
catalyst_data = best_per_ticker[best_per_ticker['Has_Catalyst'] == True]

# Group by Catalyst_Type and calculate the mean Move_Ratio
avg_move_ratio = catalyst_data.groupby('Catalyst_Type')['Move_Ratio'].mean().sort_values(ascending=False).reset_index()

plt.figure(figsize=(14, 8))

# Create a bar plot for averages
sns.barplot(
    x='Catalyst_Type',
    y='Move_Ratio',
    data=avg_move_ratio,
    palette='viridis'
)

plt.title('Average Move Ratio by Catalyst Type')
plt.xlabel('Catalyst Type')
plt.ylabel('Average Move Ratio (Predicted / Implied)')
plt.xticks(rotation=45)
plt.axhline(1.0, color='red', linestyle='--', label='Market Parity (1.0)')
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

best_per_ticker['Price_Divergence_Pct'] = (best_per_ticker['Predicted_Price'] - best_per_ticker['Current_Price']) / best_per_ticker['Current_Price']

plt.figure(figsize=(12, 7))
sns.scatterplot(
    x='Implied_Vol',
    y='Price_Divergence_Pct',
    hue='Horizon_Bucket',
    size='Expected_ROP',
    alpha=0.6,
    data=best_per_ticker
)
plt.title('Stealth Moves: Predicted Price Divergence vs. Market Fear (IV)', fontsize=14)
plt.axhline(0, color='black', linestyle='--', alpha=0.3)
plt.grid(True, alpha=0.2)
plt.show()